# **Generate Multi-Reversal ABCDEF Multi-Timestep Task**

Generates 6-stimulus reversal task sequences with **multiple reversals** for studying
whether models get faster at switching after repeated experience.

**Phase structure (10 phases total):**
- Phase 0 (pre,   4000 trials): A=100%, B=100%, C=50%, D=50%, E=0%,   F=0%
- Phase 1 (post1, 8000 trials): doubled post-reversal period
- Phases 2–9    ( 4000 trials each): alternating pre/post (4 more reversals)

**Reversal variants:**
1. **Partial** — only A and E swap; B and F keep contingencies
2. **Full**    — both A↔E and B↔F swap

Output files:
- `task_data/reversal_abcdef_multitimestep_partial_multirev.pkl`
- `task_data/reversal_abcdef_multitimestep_full_multirev.pkl`

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## Task Parameters

In [ ]:
# Stimulus identities
stimuli = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5}
stim_names = ["A", "B", "C", "D", "E", "F"]

# Phase structure: 10 phases
# Odd-index phases (1,3,5,7,9) = post-reversal contingency
# Even-index phases (0,2,4,6,8) = pre-reversal contingency
# Phase 1 is doubled (8000) to allow time to converge; remaining phases = 4000
PHASE_N_TRIALS = [4000, 8000] + [4000] * 8   # 10 phases, 44000 trials total

stim_window  = 5    # timesteps showing stimulus
reward_window = 3   # timesteps in reward-availability window
min_iti = 10
max_iti = 20

seed = 42
np.random.seed(seed)

# State map (indices for one-hot encoding)
state_map = {
    "A": 0, "B": 1, "C": 2, "D": 3, "E": 4, "F": 5,
    "reward_unknown": 6, "unrewarded": 7, "rewarded": 8, "ITI": 9
}
STATE_DIM = 10

print(f"Total phases:         {len(PHASE_N_TRIALS)}")
print(f"Trials per phase:     {PHASE_N_TRIALS}")
print(f"Total trials:         {sum(PHASE_N_TRIALS)}")
print(f"Stimulus window:      {stim_window} timesteps")
print(f"Reward window:        {reward_window} timesteps")
print(f"ITI range:            {min_iti}-{max_iti} timesteps")

---
## Reward contingency tables

In [ ]:
# Pre-reversal contingency (even phases: 0,2,4,6,8)
reward_prob_pre = {
    0: 1.0,   # A: 100%
    1: 1.0,   # B: 100%
    2: 0.5,   # C: 50%
    3: 0.5,   # D: 50%
    4: 0.0,   # E: 0%
    5: 0.0,   # F: 0%
}

# Full reversal (odd phases): A and B swap with E and F
reward_prob_post_full = {
    0: 0.0,   # A: 100% → 0%
    1: 0.0,   # B: 100% → 0%
    2: 0.5,   # C: unchanged
    3: 0.5,   # D: unchanged
    4: 1.0,   # E: 0% → 100%
    5: 1.0,   # F: 0% → 100%
}

# Partial reversal (odd phases): only A↔E swap; B and F unchanged
reward_prob_post_partial = {
    0: 0.0,   # A: 100% → 0%
    1: 1.0,   # B: unchanged 100%
    2: 0.5,   # C: unchanged
    3: 0.5,   # D: unchanged
    4: 1.0,   # E: 0% → 100%
    5: 0.0,   # F: unchanged 0%
}

print("Pre-reversal:  ", {s: f"{p*100:.0f}%" for s, p in reward_prob_pre.items()})
print("Post-full:     ", {s: f"{p*100:.0f}%" for s, p in reward_prob_post_full.items()})
print("Post-partial:  ", {s: f"{p*100:.0f}%" for s, p in reward_prob_post_partial.items()})

---
## Helper functions

In [ ]:
all_stimuli = list(stimuli.values())  # [0,1,2,3,4,5]


def sample_reward(prob):
    return 1 if np.random.rand() < prob else 0


def generate_multi_reversal_data(phase_n_trials, reward_prob_post, seed=42):
    """
    Generate trial-level data for a multi-reversal sequence.

    Odd phases use reward_prob_post; even phases use reward_prob_pre.
    Returns trial_data dict with stimuli, rewards, and masks.
    """
    np.random.seed(seed)
    trial_data = {"stimuli": [], "rewards": [], "masks": {"reversal": []}}

    for phase_idx, n_trials in enumerate(phase_n_trials):
        probs = reward_prob_pre if phase_idx % 2 == 0 else reward_prob_post
        for _ in range(n_trials):
            stim = np.random.choice(all_stimuli)
            rew  = sample_reward(probs[stim])
            trial_data["stimuli"].append(stim)
            trial_data["rewards"].append(rew)
            trial_data["masks"]["reversal"].append(phase_idx)  # raw phase index 0..N-1

    return trial_data


def expand_to_timesteps(trial_data, phase_n_trials):
    """
    Expand trial-level data to timestep-level sequences.

    Each trial entry in trial_structure has:
        reversal_phase : 0 (pre-contingency) or 1 (post-contingency)
        phase_idx      : full phase number 0..N-1
    """
    state_sequence  = []
    reward_sequence = []
    trial_structure = []

    trial_idx = 0
    timestep  = 0

    for stim, reward_avail, phase_idx in zip(
        trial_data["stimuli"],
        trial_data["rewards"],
        trial_data["masks"]["reversal"]
    ):
        reversal_phase = phase_idx % 2  # 0 = pre-contingency, 1 = post-contingency
        trial_start_ts = timestep

        # stimulus window
        stim_ts = []
        for _ in range(stim_window):
            state_sequence.append(stim)
            reward_sequence.append(0.0)
            stim_ts.append(timestep)
            timestep += 1

        # reward window
        rew_ts = []
        for _ in range(reward_window):
            state_sequence.append(state_map["reward_unknown"])
            reward_sequence.append(float(reward_avail == 1))
            rew_ts.append(timestep)
            timestep += 1

        # ITI
        iti_len = np.random.randint(min_iti, max_iti + 1)
        iti_ts  = []
        for _ in range(iti_len):
            state_sequence.append(state_map["ITI"])
            reward_sequence.append(0.0)
            iti_ts.append(timestep)
            timestep += 1

        trial_structure.append({
            "trial_idx":      trial_idx,
            "stimulus":       stim,
            "reward_available": (reward_avail == 1),
            "reversal_phase": reversal_phase,   # 0/1 binary contingency
            "phase_idx":      phase_idx,        # full phase number 0..N-1
            "trial_start":    trial_start_ts,
            "stim_window":    stim_ts,
            "reward_window":  rew_ts,
            "iti_window":     iti_ts,
            "trial_end":      timestep - 1,
        })
        trial_idx += 1

    return state_sequence, reward_sequence, trial_structure, timestep


def to_ohe(state_sequence, state_dim=STATE_DIM):
    ohe = np.zeros((len(state_sequence), state_dim), dtype=np.float32)
    for i, s in enumerate(state_sequence):
        if 0 <= s < state_dim:
            ohe[i, s] = 1.0
    return ohe


def make_phase_boundaries(trial_structure, phase_n_trials):
    """
    Build phase_boundaries dict for a multi-reversal sequence.

    Includes 'phases' list with per-phase metadata and 'reversal_points'.
    For backwards compatibility with training scripts:
        pre_reversal['end']  = end of phase 0
        post_reversal['end'] = end of entire sequence (so training loop runs to completion)
    """
    phases = []
    cum = 0
    for pi, n in enumerate(phase_n_trials):
        start_trial = cum
        end_trial   = cum + n - 1
        start_ts = trial_structure[start_trial]["trial_start"]
        end_ts   = trial_structure[end_trial]["trial_end"]
        phases.append({
            "phase_idx":   pi,
            "start":       start_ts,
            "end":         end_ts,
            "n_trials":    n,
            "contingency": "pre" if pi % 2 == 0 else "post",
        })
        cum += n

    reversal_points = [phases[i]["start"] for i in range(1, len(phases))]
    total_ts = trial_structure[-1]["trial_end"] + 1

    return {
        "reversal_points": reversal_points,
        "phases": phases,
        # backwards-compat keys
        "pre_reversal":  {"start": 0,                  "end": phases[0]["end"]},
        "post_reversal": {"start": phases[1]["start"],  "end": total_ts},
    }


def assemble_data(trial_data, state_sequence, reward_sequence,
                  trial_structure, phase_n_trials):
    state_sequence_ohe = to_ohe(state_sequence)
    phase_boundaries   = make_phase_boundaries(trial_structure, phase_n_trials)
    return {
        "state_sequence_ohe": state_sequence_ohe,
        "reward_sequence":    np.array(reward_sequence, dtype=np.float32),
        "sequence": {
            "stimuli": trial_data["stimuli"],
            "rewards": trial_data["rewards"],
            "masks":   trial_data["masks"],
        },
        "phase_boundaries": phase_boundaries,
        "trial_structure":  trial_structure,
        "state_map":        state_map,
        "trial_params": {
            "stim_window":   stim_window,
            "reward_window": reward_window,
            "min_iti":       min_iti,
            "max_iti":       max_iti,
        },
        "phase_n_trials":  phase_n_trials,
        "reversal_type":   None,  # filled in below
    }


print("Helper functions defined.")

---
## Generate: Full Reversal (multi-reversal)

In [ ]:
np.random.seed(seed)
td_full = generate_multi_reversal_data(PHASE_N_TRIALS, reward_prob_post_full, seed=seed)
ss_full, rs_full, ts_full, total_ts_full = expand_to_timesteps(td_full, PHASE_N_TRIALS)
data_full = assemble_data(td_full, ss_full, rs_full, ts_full, PHASE_N_TRIALS)
data_full["reversal_type"] = "full"

pb = data_full['phase_boundaries']
print(f"Full reversal — total timesteps: {total_ts_full:,}")
print(f"  Total trials:    {len(ts_full):,}")
print(f"  Phases: {len(pb['phases'])}")
for ph in pb['phases']:
    print(f"    Phase {ph['phase_idx']} ({ph['contingency']:4s}, {ph['n_trials']} trials): "
          f"ts {ph['start']:,}–{ph['end']:,}")
print(f"  Reversal points: {pb['reversal_points'][:5]}...")

---
## Generate: Partial Reversal (multi-reversal)

In [ ]:
np.random.seed(seed)
td_partial = generate_multi_reversal_data(PHASE_N_TRIALS, reward_prob_post_partial, seed=seed)
ss_partial, rs_partial, ts_partial, total_ts_partial = expand_to_timesteps(td_partial, PHASE_N_TRIALS)
data_partial = assemble_data(td_partial, ss_partial, rs_partial, ts_partial, PHASE_N_TRIALS)
data_partial["reversal_type"] = "partial"

print(f"Partial reversal — total timesteps: {total_ts_partial:,}")
print(f"  State sequence shape: {data_partial['state_sequence_ohe'].shape}")

---
## Verify reward contingencies

In [ ]:
for label, data in [("Full", data_full), ("Partial", data_partial)]:
    print(f"\n=== {label} reversal ===")
    phases = data['phase_boundaries']['phases']
    # Sample a subset of phases to verify
    for ph in phases[:4]:  # first 4 phases
        pi = ph['phase_idx']
        phase_trials = [t for t in data['trial_structure'] if t['phase_idx'] == pi]
        print(f"  Phase {pi} ({ph['contingency']}, {ph['n_trials']} trials):")
        for s_idx, s_name in enumerate(stim_names):
            s_trials = [t for t in phase_trials if t['stimulus'] == s_idx]
            if s_trials:
                rew_rate = np.mean([t['reward_available'] for t in s_trials])
                print(f"    {s_name}: {rew_rate*100:.1f}%")

---
## Save to pickle files

In [ ]:
# Try to find repo root (contains experiment_scripts/), then save into task_data/
repo_root = Path.cwd().resolve()
while not (repo_root / "experiment_scripts").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

output_dir = repo_root / "task_data"
output_dir.mkdir(parents=True, exist_ok=True)

path_full = output_dir / "reversal_abcdef_multitimestep_full_multirev.pkl"
path_partial = output_dir / "reversal_abcdef_multitimestep_partial_multirev.pkl"

with open(path_full, "wb") as f:
    pickle.dump(data_full, f)
print(f"Saved full reversal to:    {path_full}")

with open(path_partial, "wb") as f:
    pickle.dump(data_partial, f)
print(f"Saved partial reversal to: {path_partial}")

---
## Visualise reward contingencies across phases

In [ ]:
colors = ["#e41a1c", "#ff7f00", "#4daf4a", "#377eb8", "#984ea3", "#a65628"]
window = 200

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (label, data) in zip(axes, [("Full reversal", data_full), ("Partial reversal", data_partial)]):
    ts = data['trial_structure']
    for s_idx, s_name in enumerate(stim_names):
        s_trials = [i for i, t in enumerate(ts) if t['stimulus'] == s_idx]
        s_rewards = [ts[i]['reward_available'] for i in s_trials]
        rm = np.convolve(s_rewards, np.ones(window) / window, mode='valid')
        ax.plot(np.array(s_trials[window - 1:]), rm, color=colors[s_idx],
                label=s_name, lw=1.5)

    # Mark all reversal points (in trial units for stim A)
    a_trials = [t for t in ts if t['stimulus'] == 0]
    phases = data['phase_boundaries']['phases']
    for ph in phases[1:]:
        # find first A trial index in this phase
        a_idx = next((i for i, t in enumerate(a_trials)
                      if t['phase_idx'] == ph['phase_idx']), None)
        if a_idx is not None:
            trial_pos = [i for i, t in enumerate(ts)
                         if t['stimulus'] == 0 and t['phase_idx'] < ph['phase_idx']]
            rev_trial = len(trial_pos)
            ax.axvline(rev_trial, color='k', ls='--', lw=0.8, alpha=0.6)
            ax.text(rev_trial, 1.02, f'Rev{ph["phase_idx"]}', fontsize=7,
                    ha='center', transform=ax.get_xaxis_transform())

    ax.set_xlabel('Trial index (total)')
    ax.set_ylabel('Reward probability')
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.set_ylim(-0.05, 1.1)

plt.tight_layout()
plt.savefig(output_dir / 'multirev_reward_contingencies.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved visualisation.")